# 04. Does fertilizer explain the gap?

Notebook 03 showed Nigeria uses less fertilizer than nearly every peer and has low
cassava yields. That is two columns moving together, which proves nothing.

Here I test it with a fixed effects regression across every country with enough years of
data.

**Output:** `model_frame.csv`, `model_results.json`

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np

from src import config as C
from src import io_utils as io

pd.set_option("display.width", 170)

import statsmodels.api as sm

## 1. Why not a plain regression

Comparing countries to each other measures the difference between rich and poor farming
systems, not the effect of fertilizer.

Countries that use a lot of fertilizer also have irrigation, better seed, machinery,
storage, roads and credit. All of that would end up in the fertilizer number.

Fixed effects change the question to: when a country raises its own fertilizer use, does
its own yield go up? Nigeria in 2010 and Nigeria in 2020 share the same soil and climate,
so those differences cancel out.

## 2. Build the panel

In [2]:
wdi = io.wdi_panel()
print("\ncombined:", wdi.shape)
print("aggregates already dropped during the fetch")

  population            13,859 rows


  fert_kg_ha            10,352 rows


  cereal_yield_kg_ha    10,440 rows


  cereal_area_ha        10,495 rows


  rural_pop_pct         13,888 rows

combined: (13888, 7)
aggregates already dropped during the fetch


In [3]:
d = wdi.dropna(subset=["cereal_yield_kg_ha", "fert_kg_ha"]).copy()
d = d[(d["cereal_yield_kg_ha"] > 100) & (d["fert_kg_ha"] > 0.1)]

d["ln_yield"] = np.log(d["cereal_yield_kg_ha"])
d["ln_fert"]  = np.log(d["fert_kg_ha"])
d["ln_area"]  = np.log(d["cereal_area_ha"].clip(lower=1))
d["ln_pop"]   = np.log(d["population"].clip(lower=1))

counts = d.groupby("iso3").size()
d = d[d["iso3"].isin(counts[counts >= C.MIN_YEARS_PER_COUNTRY].index)]
d = d.dropna(subset=["ln_yield", "ln_fert", "ln_area"]).reset_index(drop=True)

print("rows:     ", len(d))
print("countries:", d["iso3"].nunique())
print("years:    ", d["year"].min(), "to", d["year"].max())

rows:      9466
countries: 178
years:     1961 to 2024


Countries with fewer than 15 years are dropped. Fixed effects work by watching a
country change over time, so a country with three rows adds nothing.

Both sides are logged, which makes the answer a percentage for a percentage. That is
comparable between Egypt at 577 kg/ha and Nigeria at 16. A plain slope would not be.

## 3. Run it

I add the controls one step at a time so I can see what each one does.

In [4]:
def fit(data, label, country_fe=True, year_fe=True):
    X = pd.DataFrame(index=data.index)
    X["ln_fert"] = data["ln_fert"].values
    X["ln_area"] = data["ln_area"].values
    X["rural_pop_pct"] = data["rural_pop_pct"].fillna(data["rural_pop_pct"].median()).values

    parts = [X]
    if country_fe:
        parts.append(pd.get_dummies(data["iso3"], prefix="c", drop_first=True, dtype=float))
    if year_fe:
        parts.append(pd.get_dummies(data["year"].astype("category"), prefix="y",
                                    drop_first=True, dtype=float))

    X = sm.add_constant(pd.concat(parts, axis=1))
    m = sm.OLS(data["ln_yield"].values, X).fit(
        cov_type="cluster", cov_kwds={"groups": data["iso3"].values})

    b, se = m.params["ln_fert"], m.bse["ln_fert"]
    print(f"{label:<28} n={int(m.nobs):>6,}  countries={data['iso3'].nunique():>4}  "
          f"b={b: .4f}  se={se:.4f}  t={b / se:5.2f}")
    return b, se, m

In [5]:
b_plain, _, _ = fit(d, "no fixed effects",  country_fe=False, year_fe=False)
b_country, _, _ = fit(d, "country only",      country_fe=True,  year_fe=False)
b, se, model  = fit(d, "country and year",    country_fe=True,  year_fe=True)

print(f"\nthe answer drops to {b / b_plain:.0%} of the plain version")

no fixed effects             n= 9,466  countries= 178  b= 0.2317  se=0.0176  t=13.15


country only                 n= 9,466  countries= 178  b= 0.1189  se=0.0180  t= 6.59


country and year             n= 9,466  countries= 178  b= 0.0864  se=0.0171  t= 5.07

the answer drops to 37% of the plain version


The plain version is nearly three times too big, and nothing about it looks wrong.
Its t statistic is over 13.

Country effects take out whatever is permanently different about each country. Year
effects take out global shocks and the worldwide rise in both yields and fertilizer.

Errors are clustered by country because the 61 Nigerian rows are not 61 independent
facts. Without clustering the range would look far too narrow.

In [6]:
lo, hi = b - 1.96 * se, b + 1.96 * se
print(f"elasticity:  {b:.4f}")
print(f"95% range:   {lo:.4f} to {hi:.4f}")
print(f"t:           {b / se:.2f}")
print()
print(f"10% more fertilizer goes with {b * 10:.2f}% more yield.")

elasticity:  0.0864
95% range:   0.0530 to 0.1198
t:           5.07

10% more fertilizer goes with 0.86% more yield.


Small, but the range never crosses zero, so it is real. Both halves matter.

## 4. Does it differ at low input levels?

I would expect fertilizer to help most where there is least of it.

In [7]:
median_fert = d["fert_kg_ha"].median()
print(f"median: {median_fert:.1f} kg/ha\n")

b_low, _, _  = fit(d[d["fert_kg_ha"] <= median_fert], "low input half")
b_high, _, _ = fit(d[d["fert_kg_ha"] >  median_fert], "high input half")

print(f"\nhigh input is {b_high / b_low:.1f}x the low input answer")

median: 59.8 kg/ha

low input half               n= 4,733  countries= 144  b= 0.0510  se=0.0155  t= 3.29


high input half              n= 4,733  countries= 130  b= 0.1251  se=0.0620  t= 2.02

high input is 2.5x the low input answer


The opposite of what I expected.

Fertilizer seems to work better where there is already plenty, which suggests it needs
the rest of the system with it: water at the right time, seed that can use it, storage,
roads, advice.

Nigeria is well below the median, so the low input number is the one that applies. That
changes the message from buy more fertilizer to fertilizer alone will not be enough.

## 5. What it means for Nigeria

In [8]:
ng = d[d["iso3"] == "NGA"].sort_values("year").tail(3)
f0 = ng["fert_kg_ha"].mean()
y0 = ng["cereal_yield_kg_ha"].mean()
ghana = d[d["iso3"] == "GHA"].sort_values("year").tail(3)["fert_kg_ha"].mean()

print(f"Nigeria now: {f0:.1f} kg/ha fertilizer, {y0:,.0f} kg/ha yield\n")

scenarios = {}
for target, name in [(ghana, "Ghana's level"), (float(median_fert), "world median")]:
    change = (np.exp(b_low * np.log(target / f0)) - 1) * 100
    scenarios[name] = {"fert_kg_ha": round(float(target), 1),
                       "yield_change_pct": round(float(change), 1),
                       "implied_yield_kg_ha": int(round(y0 * (1 + change / 100)))}
    print(f"up to {name:<16} ({target:5.1f} kg/ha)   yield {change:+5.1f}%")

Nigeria now: 10.7 kg/ha fertilizer, 1,589 kg/ha yield

up to Ghana's level    ( 34.1 kg/ha)   yield  +6.1%
up to world median     ( 59.8 kg/ha)   yield  +9.2%


Tripling Nigeria's fertilizer use goes with a single digit yield gain.

Notebook 03 found Ghana getting about four times Nigeria's cassava yield. Fertilizer
explains a small part of that. The rest is seed, disease, soil, losses after harvest, or
how the numbers are collected, and this data cannot separate them.

This is observational data, so these are associations. Not causes.

## 6. Save

In [9]:
d.to_csv(C.PROCESSED / "model_frame.csv", index=False)

results = {
    "model": "two-way fixed effects OLS, ln(cereal yield) on ln(fertilizer per ha)",
    "controls": ["ln(cereal area)", "rural population pct"],
    "standard_errors": "clustered by country",
    "n_obs": int(model.nobs),
    "n_countries": int(d["iso3"].nunique()),
    "years": [int(d["year"].min()), int(d["year"].max())],
    "elasticity": round(float(b), 4),
    "std_error": round(float(se), 4),
    "t_stat": round(float(b / se), 2),
    "ci95": [round(float(lo), 4), round(float(hi), 4)],
    "elasticity_no_fixed_effects": round(float(b_plain), 4),
    "elasticity_country_fe_only": round(float(b_country), 4),
    "elasticity_low_input": round(float(b_low), 4),
    "elasticity_high_input": round(float(b_high), 4),
    "median_fert_kg_ha": round(float(median_fert), 1),
    "nigeria_fert_kg_ha": round(float(f0), 1),
    "nigeria_cereal_yield_kg_ha": int(round(y0)),
    "scenarios": scenarios,
    "note": "observational data, associations not causes",
}

with open(C.PROCESSED / "model_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

{
  "model": "two-way fixed effects OLS, ln(cereal yield) on ln(fertilizer per ha)",
  "controls": [
    "ln(cereal area)",
    "rural population pct"
  ],
  "standard_errors": "clustered by country",
  "n_obs": 9466,
  "n_countries": 178,
  "years": [
    1961,
    2024
  ],
  "elasticity": 0.0864,
  "std_error": 0.0171,
  "t_stat": 5.07,
  "ci95": [
    0.053,
    0.1198
  ],
  "elasticity_no_fixed_effects": 0.2317,
  "elasticity_country_fe_only": 0.1189,
  "elasticity_low_input": 0.051,
  "elasticity_high_input": 0.1251,
  "median_fert_kg_ha": 59.8,
  "nigeria_fert_kg_ha": 10.7,
  "nigeria_cereal_yield_kg_ha": 1589,
  "scenarios": {
    "Ghana's level": {
      "fert_kg_ha": 34.1,
      "yield_change_pct": 6.1,
      "implied_yield_kg_ha": 1685
    },
    "world median": {
      "fert_kg_ha": 59.8,
      "yield_change_pct": 9.2,
      "implied_yield_kg_ha": 1734
    }
  },
  "note": "observational data, associations not causes"
}


---

**Next:** `05_machine_learning.ipynb` asks how predictable yield is in the first place.